In [ ]:
%pip install fastmcp pandas openai

In [12]:
from fastmcp import Client
from fastmcp.client import PythonStdioTransport
import asyncio

async def main():
    # Create a client using the PythonStdioTransport
    # Create a loop to run the client

    transport = PythonStdioTransport("mcp_server_fastmcp.py")

    async with Client(transport) as client:
        tools = await client.list_tools()
        print(" Tools are : " , [tool.name for tool in tools])
await main()

 Tools are :  ['ping', 'summarize', 'query']


#### Simulate a Simple Host Agent
Let's simulate a rule-based 'AI agent' that decides whether to use summarize or query based on user text.

In [13]:
from openai import OpenAI
import os

from dotenv import load_dotenv

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=my_api_key)

In [14]:
def decide_tool(text: str):
    # Call LLM and decide which tool to use based on the user input
    # options are "ping", "summarize" and "query"
    # For example, if the user input contains "summarize" or "overview", 
    # we can call the summarize tool.
    # If the user input contains "west", we can call the query tool 
    # with expr = "region == 'West'".
    # Otherwise, we can call the summarize tool by default.
   
   
    response = openai_client.chat.completions.create(
            model="gpt-4.1-nano", #gpt-5-nano, gpt-5-mini, gpt-5, gpt-5-pro
            messages=[
                
                {"role": "system", 
                "content": '''You are a helpful assistant. 
                            decide which tool to use based on the user input
                            options are "ping", "summarize" and "query"
                            For example, if the user input contains "summarize" or "overview", 
                            we can call the summarize tool.
                            If the user input contains "west", we can call the query tool 
                            with expr = "region == 'West'".
                            Otherwise, we can call the summarize tool by default.
                            return the tool name and parameters in the following format:
                            tool_name
                            '''},
                {"role": "user", "content": text}
            ]
        )
    return_value = response.choices[0].message.content.strip()
    if "summarize" in return_value:
        return "summarize", {}
    if "query" in return_value and "west" in text.lower():
        return "query", {"expr": "region == 'West'"}
    return "summarize", {}

    
    return return_value, {} # Assuming the tool name is in the first line and parameters are empty for simplicity    
    

async def run_agent(user_input, client):
    tool, params = decide_tool(user_input) #"summarize", {}
    #tool = "summarize"
    #params = {}
    print(f"Agent decided to use '{tool}'")

    # API for fastmcp 2.12.5
    result = await client.call_tool(tool, params)
    # result = await client.call_tool("query", params)

    print("Result:", result, "\n")

In [15]:
# create a connection to your MCP server
transport = PythonStdioTransport("mcp_server_fastmcp.py")

async with Client(transport) as client:
    await run_agent("Give me a summary of the dataset", client)

Agent decided to use 'summarize'
Result: CallToolResult(content=[TextContent(type='text', text='{"rows":10,"columns":["order_id","region","sales","quarter","rep"],"numeric_stats":{"order_id":{"count":10.0,"unique":"","top":"","freq":"","mean":105.5,"std":3.0276503540974917,"min":101.0,"25%":103.25,"50%":105.5,"75%":107.75,"max":110.0},"region":{"count":10,"unique":4,"top":"West","freq":4,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""},"sales":{"count":10.0,"unique":"","top":"","freq":"","mean":1235.0,"std":536.4751210965477,"min":600.0,"25%":825.0,"50%":1175.0,"75%":1450.0,"max":2200.0},"quarter":{"count":10,"unique":4,"top":"Q1","freq":3,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""},"rep":{"count":10,"unique":10,"top":"Alex","freq":1,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""}}}', annotations=None, meta=None)], structured_content={'rows': 10, 'columns': ['order_id', 'region', 'sales', 'quarter', 'rep'], 'numeric_stats': {'order_

In [16]:
# create a connection to your MCP server
transport = PythonStdioTransport("mcp_server_fastmcp.py")

async with Client(transport) as client:
    await run_agent("Give me a summary of the dataset", client)
    await run_agent("Show West region sales > 1500", client)

Agent decided to use 'summarize'
Result: CallToolResult(content=[TextContent(type='text', text='{"rows":10,"columns":["order_id","region","sales","quarter","rep"],"numeric_stats":{"order_id":{"count":10.0,"unique":"","top":"","freq":"","mean":105.5,"std":3.0276503540974917,"min":101.0,"25%":103.25,"50%":105.5,"75%":107.75,"max":110.0},"region":{"count":10,"unique":4,"top":"West","freq":4,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""},"sales":{"count":10.0,"unique":"","top":"","freq":"","mean":1235.0,"std":536.4751210965477,"min":600.0,"25%":825.0,"50%":1175.0,"75%":1450.0,"max":2200.0},"quarter":{"count":10,"unique":4,"top":"Q1","freq":3,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""},"rep":{"count":10,"unique":10,"top":"Alex","freq":1,"mean":"","std":"","min":"","25%":"","50%":"","75%":"","max":""}}}', annotations=None, meta=None)], structured_content={'rows': 10, 'columns': ['order_id', 'region', 'sales', 'quarter', 'rep'], 'numeric_stats': {'order_